# 19.14 同一個世界，怎樣避免三份資料互相洩題？


假設一筆事件是「起始2，後來設成5，再設成1」。B用它學最後狀態；C可能用它學「這是追蹤任務」；Auto又用它記不同輪數能否答對。如果各自隨機切資料，同一事件便可能在B的訓練、C的測試中同時出現，最後的整合分數不再代表新情境。

因此先給**事件家族**一個固定識別碼，再共同決定切分。家族包含同一底層事件的措辭改寫、不同問題及候選重排；從家族衍生的B、C與路由紀錄，全部繼承相同split。題目ID標一筆具體輸入，家族ID標它和哪些題有共同來源，兩者角色不同。

下例只示範穩定分配，不訓練模型。真正的比例與最終題數會寫在資料manifest；相同家族不管衍生幾筆都不換組。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import hashlib


def split_of(family_id):
    bucket = int(hashlib.sha256(family_id.encode()).hexdigest()[:8], 16) % 10
    return "train" if bucket < 7 else "validation" if bucket < 9 else "test"


for variant in ["原句", "改問資訊是否充分", "候選倒序"]:
    print(variant, split_of("state:start2:set5:set1"))

三行應相同。這能防同家族跨組，卻不自動證明測到新推理能力：如果所有問題仍套同一句型，模型可能只熟悉語法。報告還會分開列未見事件組合、實體／措辭變化及更長事件序列；每種測試說明哪些關係見過、哪些沒有。有限語法任務可先驗收有限能力，不能改名成一般中文理解。

validation 也有不同用途。calibration 部分選機率溫度，policy 部分選輪數與門檻；最後 test 封存，到設定凍結後才評分。若已看過某批test的錯題並修訓練，那批只能作開發診斷，新的最後評估另留資料。不能不斷查同一考卷，仍稱一次最後測試。

每個版本保存生成器、seed、世界規則、家族與題目ID、欄位說明、檔案指紋及切分統計。合成資料不等於免查洩漏；同義重排、對稱算式或相同事件的等價編碼，也要先定義如何合併家族。

練習：`2+3`與`3+2`要不要同組？若本次希望測未見加數組合，就應用排序後的加數建立共同家族。這是測試目標決定的契約，不能等看到成績才改。
